## Before running: paths, timing caches, and output replacement

See the [raw-to-RF guide](README.md) and [platform and data-safety notes](docs/platforms_and_data_safety.md). Start this kernel in the repository directory so `./file_names.json` resolves. Change `base_dir`, recording identifiers, probe, and headplate to your data; `/mnt/...` is an example Linux mount. On native Windows use a raw string such as `r"D:\recordings\mouse_01"`; on macOS/Linux use the actual mounted path. Preserve directory capitalization.

- With the supplied `file_names.json`, the preprocessing caches are `data/session_info.json`, `data/sync_data.json`, and `data/interval_table.csv`. Copied `session_info.json` can retain another machine's `session_info["base_path"]` and `session_name`; inspect those paths after moving data. Changing only `base_dir` does not rewrite this cache.
- The current `gen_recording_interval_table(...)` call leaves `is_overwrite_interval_table`, `is_overwrite_session_file`, and `is_overwrite_sync_file` at `False`. An existing interval table skips the generator entirely. Changing a session/sync overwrite flag alone therefore does not regenerate it. Preserve existing files and any edited interval table before deliberately regenerating the specific derived cache; do not delete a timing-edit database to fix a path.
- Camera timing uses `data/camera_frame_times.npy` first, then saved midpoint data in `data/sync_data.json`, then raw ADC. Check `ttl_qc["source_path"]`; changing channel settings does not invalidate saved timing.
- `is_save=True` replaces `data/tuning_curves/ProbeA/tuning_curves.tc`, `hd_cells_1.npy`, and `hd_cells_2.npy` for the chosen probe. Use `is_save=False` for calculation without those final writes, or supply `save_path` in a new directory to preserve the curve and both unit lists. Preprocessing can still create missing caches when final-result saving is disabled.


In [8]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from tqdm import tqdm
import numpy as np
import pandas as pd

from Utils.load_files import get_interval_pairs
from Utils.json_tools import read_formatted_json
from Utils.tuning_curve_utils import get_exposure_timestamps, make_head_direction_tsd, tuning_curve
from Utils.recording import gen_recording_interval_table

file_names = read_formatted_json("./file_names.json")
session_info_filename: str = file_names["session_info_filename"]
head_direction_filename: str = file_names["head_direction_filename"]
interval_table_filename: str = file_names["interval_table_filename"]
kilosort_info_filename: str = file_names["kilosort_info_filename"]


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
base_dir = r"/mnt/senzailab/Kai/#Recording/m15"

date: str | int = "260630"
multi_recording: bool = True
num_of_rec_list: list[int] = [1, ]
headplate_name: str = 'hp4'
probe_name: str = "A"

phase_key = "baseline"
num_of_bins_in_hd: int = 180
num_shuffle: int = 1000
shuffle_seed: int = 0

camera_input_channel: int = 1

for num_of_rec in num_of_rec_list:

    with tqdm(total=5, desc="Reading files", unit="%",
              bar_format="{l_bar}{bar}| {n}/{total} [{percentage:3.0f}%]") as pbar:
        subfolder_filler = f"{date}_{num_of_rec}"
        session_dir = f"{base_dir}/{date}/{subfolder_filler}" if multi_recording else f"{base_dir}/{date}"

        gen_recording_interval_table(base_dir=session_dir, multi_recording=multi_recording,
                                     camera_input_channel=camera_input_channel, )

        data_dir: str = f"{session_dir}/data"
        kilosort_dir = next((Path(session_dir) / "kilosort" / f"Probe{probe_name}").glob("kilosort_*"))

        session_info: dict = read_formatted_json(f"{data_dir}/{session_info_filename}.json")["session_info"]
        pbar.update(1)

        interval_table = pd.read_csv(f"{data_dir}/{interval_table_filename}.csv")
        interval_pairs_all = np.asarray(
            get_interval_pairs(interval_table,
                               phase_key=phase_key), dtype=float)
        pbar.update(1)

        hd_content = read_formatted_json(f"{data_dir}/processed/{head_direction_filename}.json")
        hd_raw = hd_content[headplate_name].get('head_direction_deg')
        hd = np.asarray(hd_raw, dtype=float)
        hd = hd % 360
        hd_frames = np.asarray(hd_content[headplate_name].get('frames'), dtype=int)
        pbar.update(1)

        exposure_timestamps, adc_time_origin_s, ttl_qc = get_exposure_timestamps(
            session_info=session_info,
            data_dir=data_dir,
        )
        pbar.update(1)

        assert np.array_equal(hd_frames, np.arange(len(hd_frames))), (
            "Motive frame IDs must be zero-based and continuous."
        )

        if len(hd_frames) == len(exposure_timestamps) + 1:
            print(f"Dropping trailing Motive frame: {hd_frames[-1]}")
            hd = hd[:-1]
            hd_frames = hd_frames[:-1]
        elif len(hd_frames) != len(exposure_timestamps):
            raise ValueError(
                f"Motive frame/TTL count mismatch: {len(hd_frames)} frames vs {len(exposure_timestamps)} exposure pulses.")

        assert len(hd) == len(hd_frames)

        # head_direction.json must already use the GUI convention: 0 degrees up,
        # positive counter-clockwise. This notebook only applies modulo 360.
        HD_tsd, feature_fs_hz = make_head_direction_tsd(exposure_timestamps, hd_frames, hd)
        ttl_qc["motive_frame_count"] = int(len(hd_frames))
        pbar.update(1)

    interval_pairs = interval_pairs_all[[0]]

    hd_tuning_curves_A_formatted = tuning_curve(base_dir=session_dir,
                                                kilosort_dir=kilosort_dir,
                                                probe_name=probe_name,
                                                interval_pairs=interval_pairs,
                                                HD_tsd=HD_tsd,
                                                adc_time_origin_s=adc_time_origin_s,
                                                num_of_bins_in_hd=num_of_bins_in_hd,
                                                num_shuffle=num_shuffle,
                                                shuffle_seed=shuffle_seed,
                                                is_save=True,
                                                feature_fs_hz=feature_fs_hz,
                                                metadata={
                                                    "epoch": phase_key,
                                                    "headplate": headplate_name,
                                                    "ttl_qc": ttl_qc,
                                                },
                                                )


Classifying HD cells: 100%|██████████| 147/147 [00:16<00:00,  8.91unit/s]
